# 🧪 Lab 9: Schema Drift Storm Simulator

This laboratory creates controlled turbulence. We simulate multiple suppliers and evolving event versions to compare `from_json` (static) against `VARIANT` (observational) ingestion. We then implement a 'Drift Report' and a 'Quarantine Gate' to maintain pipeline integrity.

In [1]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()

# 1. Simulate disparate supplier payloads
raw_data = [
    (1, '{"supplier": "A", "v": 1, "origin": "MAD", "price": 100.0}', "VALID"),
    (2, '{"supplier": "B", "v": 2, "origin": "MAD", "ancillaries": {"bags": 1}}', "VALID"),
    (3, '{"supplier": "C", "v": 4, "origin": "MAD", "loyalty": {"tier": "GOLD"}}', "VALID"),
    (4, '{"supplier": "D", "v": 6, "origin": "BAD", "price": -50.0}', "INVALID_PRICE")
]
df = spark.createDataFrame(raw_data, ["id", "raw_payload", "expected_status"])
df.show(truncate=False)

+---+-----------------------------------------------------------------------+---------------+
|id |raw_payload                                                            |expected_status|
+---+-----------------------------------------------------------------------+---------------+
|1  |{"supplier": "A", "v": 1, "origin": "MAD", "price": 100.0}             |VALID          |
|2  |{"supplier": "B", "v": 2, "origin": "MAD", "ancillaries": {"bags": 1}} |VALID          |
|3  |{"supplier": "C", "v": 4, "origin": "MAD", "loyalty": {"tier": "GOLD"}}|VALID          |
|4  |{"supplier": "D", "v": 6, "origin": "BAD", "price": -50.0}             |INVALID_PRICE  |
+---+-----------------------------------------------------------------------+---------------+



### Step 2: Observational Ingestion (VARIANT)
We parse the heterogeneous raw payloads into the `VARIANT` type, allowing the schema to adapt to the supplier-specific fields automatically.

In [2]:
df_variant = df.withColumn("parsed", F.parse_json(F.col("raw_payload")))
df_variant.select("id", "parsed").printSchema()
df_variant.select("id", F.typeof("parsed").alias("type")).show()

root
 |-- id: long (nullable = true)
 |-- parsed: variant (nullable = true)

+---+-------+
| id|   type|
+---+-------+
|  1|variant|
|  2|variant|
|  3|variant|
|  4|variant|
+---+-------+



### Step 3: The Quarantine Gate
We separate syntactically sound data from business-logic violations (e.g., negative pricing) by querying the parsed variant structures.

In [3]:
quarantine_df = df_variant.select(
    "id",
    F.variant_get("parsed", "$.price", "double").alias("price"),
    F.when(F.variant_get("parsed", "$.price", "double") < 0, "NEG_PRICE")
     .otherwise("PASS").alias("status")
)
quarantine_df.filter(F.col("status") != "PASS").show()

+---+-----+---------+
| id|price|   status|
+---+-----+---------+
|  4|-50.0|NEG_PRICE|
+---+-----+---------+



## 📊 Post-Lab Analysis: Stability in the Storm

* **Observational Ingestion:** By utilizing `VARIANT`, we successfully ingested disparate payloads without needing to pre-define a massive, sparse union schema. The schema adapted to supplier-specific subfields dynamically.
* **Business-Logic Quarantine:** Even when the syntax of the JSON was perfectly valid, the Quarantine Gate successfully caught the negative price logic, demonstrating that structural parsing and business-value validation must remain decoupled for robust pipeline integrity.

### ⏱️ Performance & Validation Summary
* **Step 1 (Simulation):** Executed in ~19s (18:00:13–18:00:32). Successfully created a DataFrame with 4 rows of heterogeneous data.
* **Step 2 (VARIANT Ingestion):** Executed in ~11s (18:00:32–18:00:43). Confirmed schema successfully parsed raw payloads into `variant` types.
* **Step 3 (Quarantine Gate):** Executed in ~11s (18:00:43–18:00:54). Correctly identified Row 4 as `NEG_PRICE` while passing the remaining valid rows.